# Three arms, one question

**Elena · Replication · Bilateral** — what changes between them, and what the data will support
being said about it.

This replaces a notebook that was copied from `analysis/elena_replication/` and described that
analysis rather than this one.

## Why three arms and not two

Each arm changes **one thing** from the one before it, which is the only reason the comparison can
be read at all:

```
Elena         SO_396 broad          ·  original masks  ·  5 / 9 / 10 PCs
Replication   SO_396 broad          ·  rebuilt masks   ·  5 / 4 / 3  PCs
Bilateral     bilateral sensorineural · rebuilt masks  ·  5 / 4 / 3  PCs
```

So **Elena → Replication** isolates the mask and PC corrections, and **Replication → Bilateral**
isolates the phenotype decision. Anything that moves between two arms has one candidate cause.

## Making the three comparable

Elena's pipeline emits no per-gene omnibus. Its `Cauchy` rows combine annotations *inside* one cell,
leaving about nine per gene — so they answer a different question from ours. Hers is therefore
recomputed here with the same ACAT over the same nine cells, and her `ALL` mask is dropped since the
other two never ran it.

Her table also carries two defects the replication documented: rows whose columns are shifted, from
three files that lost their header, and a `-` pseudo-gene pooling every variant VEP could not name.
Both are dropped rather than averaged in — the count appears below.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from scipy.stats import fisher_exact

HERE = Path.cwd().parents[1]          # the analysis root
r = json.load(open(HERE / "phase_5/results/09_three_way.json"))

ARMS = ["Elena", "Replication", "Bilateral"]
COHORTS = ["combined", "EUR", "AFR"]

pd.DataFrame(r["design"]).T

,phenotype,masks,pcs,cases
Elena,SO_396 (broad),original,5 / 9 / 10,{'combined': 6712}
Replication,SO_396 (broad),rebuilt,5 / 4 / 3,{'combined': 6752}
Bilateral,bilateral sensorineural,rebuilt,5 / 4 / 3,{'combined': 3164}


In [2]:
print(f"rows dropped from Elena's table: {r['her_rows_dropped']:,}")

rows dropped from Elena's table: 699,784


## 1. No arm has a significant gene

The headline first, because everything after it is about the ranking *underneath* a null result.

In [3]:
rows = []
for c in COHORTS:
    for a in ARMS:
        v = r["cohorts"][c]["arms"][a]
        rows.append({"cohort": c, "arm": a, "genes": v["genes"],
                     "min omnibus p": v["min_omnibus_p"],
                     "bar 0.05/genes": v["bar"],
                     "significant": v["significant"]})
pd.DataFrame(rows).set_index(["cohort", "arm"])

genes  min omnibus p  bar 0.05/genes  significant
cohort   arm                                                           
combined Elena        19042       0.000005        0.000003        False
         Replication  17943       0.000030        0.000003        False
         Bilateral    17941       0.000021        0.000003        False
EUR      Elena        18939       0.000014        0.000003        False
         Replication  17928       0.000069        0.000003        False
         Bilateral    17921       0.000014        0.000003        False
AFR      Elena        18680       0.000035        0.000003        False
         Replication  17789       0.000022        0.000003        False
         Bilateral    17778       0.000009        0.000003        False

## 2. Known deafness genes at the top

The ClinGen Hearing Loss panel's Definitive and Strong genes — 100 of them, about 93 tested in each
arm. If a ranking carries biology, they sit near its top more often than chance allows.

![three arms](../../figures/three_way_clingen.png)

In [4]:
rows = []
for c in COHORTS:
    for a in ARMS:
        v = r["cohorts"][c]["arms"][a]
        rows.append({"cohort": c, "arm": a,
                     "in top 50": v["clingen_in_top50"],
                     "expected": v["clingen_expected"],
                     "p": v["clingen_p"],
                     "genes": " · ".join(v["clingen_genes"])})
pd.DataFrame(rows).set_index(["cohort", "arm"])

in top 50  expected         p  \
cohort   arm                                          
combined Elena                2      0.24  0.024834   
         Replication          5      0.26  0.000006   
         Bilateral            0      0.26  1.000000   
EUR      Elena                4      0.25  0.000106   
         Replication          2      0.26  0.027755   
         Bilateral            0      0.26  1.000000   
AFR      Elena                1      0.25  0.221106   
         Replication          0      0.26  1.000000   
         Bilateral            0      0.26  1.000000   

                                                    genes  
cohort   arm                                               
combined Elena                                COCH · SIX1  
         Replication  COCH · GJB3 · MYO6 · SIX1 · TMPRSS3  
         Bilateral                                         
EUR      Elena               COCH · SIX1 · TMPRSS3 · TPRN  
         Replication                          SIX1 · TPRN  
         Bilateral                                         
AFR      Elena                                       GJB3  
         Replication                                       
         Bilateral

### What this supports

**Against chance**, which is a test each arm passes or fails on its own:

- `Replication / combined` is the strongest single observation — 5 genes against 0.26 expected,
  p = 5.9 × 10⁻⁶.
- `Elena / EUR` also clears it, 4 against 0.25, p = 1.1 × 10⁻⁴.
- `Bilateral` shows nothing in any cohort.

So at least two arms carry known biology in their ranking. That is worth having: it means these
orderings are not pure noise, which a null headline might otherwise suggest.

## 3. What this does NOT support — arm against arm

The figure invites a story: the corrections took combined from 2 to 5, the restriction took it to 0.
**The data do not support any of it.** Fisher's exact on the 50 slots, between every pair:

In [5]:
rows = []
for c in COHORTS:
    a = r["cohorts"][c]["arms"]
    for x, y in [("Elena", "Replication"), ("Replication", "Bilateral"),
                 ("Elena", "Bilateral")]:
        hx, hy = a[x]["clingen_in_top50"], a[y]["clingen_in_top50"]
        p = fisher_exact([[hx, 50 - hx], [hy, 50 - hy]])[1]
        rows.append({"cohort": c, "comparison": f"{x} vs {y}",
                     "counts": f"{hx} vs {hy}", "p": round(p, 3),
                     "distinguishable": p < 0.05})
pd.DataFrame(rows).set_index(["cohort", "comparison"])

counts      p  distinguishable
cohort   comparison                                              
combined Elena vs Replication      2 vs 5  0.436            False
         Replication vs Bilateral  5 vs 0  0.056            False
         Elena vs Bilateral        2 vs 0  0.495            False
EUR      Elena vs Replication      4 vs 2  0.678            False
         Replication vs Bilateral  2 vs 0  0.495            False
         Elena vs Bilateral        4 vs 0  0.117            False
AFR      Elena vs Replication      1 vs 0  1.000            False
         Replication vs Bilateral  0 vs 0  1.000            False
         Elena vs Bilateral        1 vs 0  1.000            False

**Not one pair is distinguishable.** The closest is `Replication 5 vs Bilateral 0` at p = 0.056,
and that test is already optimistic: the arms share people, genes and controls, so they are not
independent and a test that assumes they are overstates the evidence.

The honest statement is therefore narrower than the figure looks:

> Two arms show enrichment against chance. No arm is shown to differ from another. The counts are
> 0 to 5 against an expectation of 0.26, and differences among such small numbers are not
> measurable here.

This matters because the available stories are tempting in both directions — "our corrections found
the biology" and "the restriction destroyed it" — and neither is supported. The phenotype decision
in particular was tested properly by the two controls in `phase_5/power_control/` and
`phase_5/complement/`, and came out exonerated.

## 4. How much of each top 50 is shared

In [6]:
rows = []
for c in COHORTS:
    for pair, n in r["cohorts"][c]["top50_overlap"].items():
        rows.append({"cohort": c, "pair": pair, "shared of 50": n})
pd.DataFrame(rows).pivot(index="pair", columns="cohort",
                         values="shared of 50")[COHORTS]

cohort,combined,EUR,AFR
pair,,,
Elena vs Bilateral,6,6,5
Elena vs Replication,18,18,18
Replication vs Bilateral,9,11,6


Elena and Replication share about 18 of 50 in every cohort; either against Bilateral shares 5 to
11. **Changing who counts as a case moves the ranking more than the mask and PC corrections did** —
which is a statement about rank overlap, not about which ranking is better.

## 5. The top five of each arm

In [7]:
pd.DataFrame({(c, a): r["cohorts"][c]["arms"][a]["top5"]
              for c in COHORTS for a in ARMS}).T

0      1       2        3       4
combined Elena        TMC3-AS1   PSG8  PTPN23  EFCAB12     MCC
         Replication      PSG8  PSMA7    BPTF     SIX1  LSMEM2
         Bilateral       PCBD1  TJAP1  LSMEM2   FBXO25     MCC
EUR      Elena        TMC3-AS1   SIX1    CHGB  SLCO1B7    BPTF
         Replication   ATP13A4   CHGB    FGD3     SIX1   PSMA7
         Bilateral      FBXO25  PCBD1   TJAP1    OXGR1    ODR4
AFR      Elena           MLLT6   ELOB   IGSF9     PIGH    FBN3
         Replication       CFH  MLLT6   TMOD2     ACLY    MAP9
         Bilateral       IGSF9  OR6Y1  RAET1E     RFC1  CREBZF

## What to take away

1. **No arm has a significant gene.** Three phenotype-and-method combinations, none crosses the bar.
2. **The rankings are not noise.** Known deafness genes cluster at the top of two of the three arms
   more than chance allows, which is the one positive result in this work.
3. **No arm is shown to be better than another.** Every pairwise difference is within what these
   counts can resolve, and saying otherwise would be reading the figure rather than the test.
4. **The cohort is the limit.** 3,164 or 6,752 cases is small for rare-variant burden, and that is
   what the three arms agree on.